In [ ]:
from aspire.samples import PTMCMCSamples
from aspire.utils import AspireFile
from aspire_analysis_tools.utils import compute_rhat
import matplotlib.pyplot as plt
import arviz

In [ ]:
result_files = {
    "gaussian": "/home/michael/git_repos/aspire-analyses/toy_examples/outdir/gaussian_comparison_fixed/15d/aspire_smc_results.h5",
}

In [ ]:
for result_name, result_file in result_files.items():
    with AspireFile(result_file) as f:
        ptmcmc_samples = PTMCMCSamples.load(f, "ptmcmc_samples")
        ptmcmc_samples_alt = PTMCMCSamples.load(f, "ptmcmc_samples_alt")
        ptmcmc_samples_flow = PTMCMCSamples.load(f, "ptmcmc_samples_flow")

    rhat_a = compute_rhat(ptmcmc_samples.cold_chain())
    rhat_b = compute_rhat(ptmcmc_samples_alt.cold_chain())
    rhat_flow = compute_rhat(ptmcmc_samples_flow.cold_chain())

    print(f"R-hat for PTMCMC A: {rhat_a}")
    print(f"R-hat for PTMCMC B: {rhat_b}")
    print(f"R-hat for PTMCMC C (flow): {rhat_flow}")

    for title, chain in zip(
        ["PTMCMC A", "PTMCMC B", "PTMCMC C (flow)"],
        [ptmcmc_samples, ptmcmc_samples_alt, ptmcmc_samples_flow],
    ):
        posterior_chain = chain.cold_chain()
        var_names = posterior_chain.parameters
        # Compute rhat for each temperature

        rhat_values = {}

        for i, beta in enumerate(chain.betas):
            print(f"Beta: {beta}")
            chain_at_temp = chain.at_temperature(i)

            arviz_data = arviz.from_dict(
                {"posterior": chain_at_temp.chain.transpose(1, 0, 2)}
            )
            rhat = arviz.rhat(arviz_data)
            rhat_values[beta] = dict(zip(var_names, rhat["posterior"].values))

        # Plot R-hat values for each parameter across temperatures
        plt.figure(figsize=(10, 6))
        for param in var_names:
            rhat_param = [rhat_values[beta][param] for beta in chain.betas]
            plt.plot(chain.betas, rhat_param, marker="o", label=param)
        plt.axhline(1.1, color="red", linestyle="--", label="R-hat = 1.1")
        plt.xscale("log")
        plt.xlabel("Inverse Temperature (beta)")
        plt.ylabel("R-hat")
        plt.title("R-hat Values Across Temperatures for " + title)
        plt.legend()
        plt.grid()
        plt.show()